# Task-grouped local model comparison

This notebook reruns the checked-in CPU experiment and reads the machine-readable output. It classifies the **CTH release's trace-source labels**, not verified operators. Human terminal capture and Agent tool-call capture are perfectly aligned with the label.

Protocol: five-fold `GroupKFold` by `benchmark/task_id`; equal task weight within each benchmark, then equal benchmark weight; 5,000 benchmark-stratified task-cluster bootstrap replicates over fixed out-of-fold predictions. Brier and 10-bin ECE are computed on raw scores because no calibration model was fitted.

Jev and OpenAI Decisions are listed as not run: this environment had no API credential or verifiable free allowance, and no paid API request was made.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
assert (ROOT / "scripts/compare_trace_mode_models.py").exists(), "Open this notebook from the repository root"
subprocess.run([sys.executable, "scripts/compare_trace_mode_models.py"], cwd=ROOT, check=True)
result = json.loads((ROOT / "results/local_model_comparison.json").read_text())
assert result["data"]["records"] == 177
assert result["data"]["task_groups"] == 40


In [ ]:
rows = []
for name, metrics in result["models"].items():
    row = {"model": name}
    for key in ("balanced_accuracy", "macro_f1", "roc_auc", "brier", "ece_10_bin"):
        ci = metrics["95_task_cluster_ci"][key]
        row[key] = metrics[key]
        row[key + "_95_ci"] = ci
    rows.append(row)
summary = pd.DataFrame(rows)
display(summary)


## Interpretation

Command-name TF-IDF + logistic regression has the highest balanced accuracy and ROC-AUC among these local baselines. ExtraTrees has the lowest ECE point estimate, while none of the raw scores has been calibrated. This ranking is conditional on 40 task groups and the current capture-path confound; it does not establish real-world actor identification.


In [ ]:
display(Image(filename=str(ROOT / "paper/figures/figureA7_local_model_comparison.png")))
print("API comparators:", result["api_comparators"])
